# Gateway tracking with an uncertain halo observer

Compare Gateway orbit estimation over one observer period using (1) exact observer knowledge and (2) an observer orbit estimated from lunar surface observations. Both cases use the same truth trajectories, Gateway prior and noisy Gateway measurements.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the project checkout")
sys.path.insert(0, str(ROOT / "src"))
from halo_tracking.study import load_config
cfg = load_config(ROOT / "configs/baseline.json")
cfg["simulation"]["periods"] = 1.0  # One nominal observer period


## 1. Matched comparison

| Assumption | Perfect observer | LSO-estimated observer |
|---|---|---|
| Observer state supplied to Gateway estimator | Exact true state | State estimated from LSO angles |
| Initial observer uncertainty | Zero | 1 km and 0.01 m/s per axis, 1σ |
| Initial Gateway uncertainty | 1 km and 0.01 m/s per axis, 1σ | Same |
| Observer-to-Gateway camera | 1 arcsecond per angle | Same measurements and noise |
| LSO | Not needed | 30° N at 270° E and 90° E, 85° S at 90° E, 1 arcsecond per angle |

These are the baseline settings. The configuration table below displays the values actually used. Both optical links use **angles only** in this focused comparison. The cadence is 30 minutes. Earth/Moon occultation is included for Gateway observations. The LSO uses the configured elevation mask. Sun 20° / Earth 10° keep-outs and the configured range cap apply to both links. Optical brightness and camera attitude errors are not modeled.

The true observer remains on the supplied initial-value trajectory in both cases. The imperfect case introduces uncertainty in our **knowledge** of it, not a different physical orbit.

In [ ]:
from IPython.display import HTML, Markdown, display
settings = [
    ("Duration", f"1 observer nominal period = {cfg['orbit']['period_tu'] * cfg['time_unit_s'] / 86400:.2f} days"),
    ("Observer initial position / velocity 1σ", f"{cfg['estimation']['initial_position_sigma_km']:g} km / {1000*cfg['estimation']['initial_velocity_sigma_km_s']:g} m/s per axis"),
    ("Gateway initial position / velocity 1σ", f"{cfg['comparison']['gateway_position_sigma_km']:g} km / {1000*cfg['comparison']['gateway_velocity_sigma_km_s']:g} m/s per axis"),
    ("LSO angular noise 1σ", f"{cfg['measurement']['angle_sigma_arcsec']:g} arcsec per angle"),
    ("Gateway camera angular noise 1σ", f"{cfg['comparison']['gateway_camera_sigma_arcsec']:g} arcsec per angle")]
display(HTML("<table>"+"".join(f"<tr><th>{name}</th><td>{value}</td></tr>" for name,value in settings)+"</table>"))

In [ ]:
from halo_tracking.gateway_comparison import run_gateway_comparison, save_comparison
result = run_gateway_comparison(cfg)
report = save_comparison(result, ROOT / "results/gateway_comparison")

## 2. Geometry and what the LSO knows

The first figure spans exactly one observer nominal period (Gateway repeats at its own period). It is context: both true trajectories are identical across the two estimation cases. The dots mark their initial states. The axis scale is in thousands of kilometers, with equal physical scale on all three axes.

The second figure shows the observer's estimated-state error and its local covariance prediction. The shaded intervals contain observations from at least one LSO. The three-site network is evaluated at every scheduled epoch. **RSS σ is a covariance summary, not a guaranteed bound on the error magnitude.**

In [ ]:
from halo_tracking.plotting import geometry, observer_knowledge, gateway_comparison_figures, state_histories, export
fig_geometry = geometry(result)
export(fig_geometry, ROOT / "results/gateway_comparison/geometry.png")

In [ ]:
fig_observer = observer_knowledge(result)
export(fig_observer, ROOT / "results/gateway_comparison/observer.png")

## Observer position, velocity and acceleration

The solid curve is the perfect observer state (truth). The dashed curve is the LSO-predicted state. Components use Moon-centered inertial axes. These curves nearly overlap at orbit scale. The error plot above resolves their differences.

In [ ]:
fig_observer_states = state_histories(result, 'observer')
export(fig_observer_states, ROOT / 'results/gateway_comparison/observer_states.png')

## Gateway position, velocity and acceleration

Compare the true Gateway trajectory with the two tracked trajectories: one uses the perfect observer and one uses the LSO-predicted observer. Their small separation is quantified by the error plot below.

In [ ]:
fig_gateway_states = state_histories(result, 'gateway')
export(fig_gateway_states, ROOT / 'results/gateway_comparison/gateway_states.png')

## 3. What changes in Gateway tracking

**Actual-error plot:** distance between the estimated and true Gateway state in this one shared noise realization. Lower is better. It is an actual orbit estimate, not just a pointing-error projection.

**Predicted-uncertainty plot:** square root of the position/velocity covariance-block trace. Lower means the estimator predicts better knowledge. These are local linearized predictions. They can be optimistic if the approximation is inaccurate.

In the imperfect case, the estimator carries observer–Gateway cross-covariance through time. Gateway measurements do not refine the observer: its state knowledge comes from the LSO. LSO updates can also correct Gateway indirectly through the retained correlation.

In [ ]:
figures = gateway_comparison_figures(result)
fig_errors = figures["errors"]
export(fig_errors, ROOT / "results/gateway_comparison/errors.png")
import matplotlib.pyplot as plt
plt.close(figures["uncertainty"])

In [ ]:
figures_uncertainty = gateway_comparison_figures(result)
plt.close(figures_uncertainty["errors"])
fig_uncertainty = figures_uncertainty["uncertainty"]
export(fig_uncertainty, ROOT / "results/gateway_comparison/uncertainty.png")

## Model and limitations

Propagation uses the Earth–Moon CR3BP. The plots transform normalized rotating states into Moon-centered inertial position, velocity and acceleration. Acceleration is calculated from each state using the dynamics. It is not independently estimated.

The imperfect case retains observer–Gateway cross-covariance. LSO measurements update observer knowledge. Gateway measurements update Gateway without refining the observer. Initial covariances are diagonal, with zero observer–Gateway cross-covariance. The forecast uses zero innovations along the nominal trajectories. The actual estimate uses noisy measurements.

The supplied `SNRHL2` state is a southern NRHO initial condition rounded to four decimals. Its one-period closure miss is about 99 km. This study assumes no stationkeeping or unmodeled forces. Local filter covariance can be overconfident near nonlinear updates, so predicted uncertainty is a conditional comparison rather than a validated error bound.

Gateway's initial state comes from [the earlier cislunar project](https://github.com/scottblender/Gradient-Free-Optimization-for-Cislunar-SDA-2/blob/main/scripts/build_target_case_database.m). Both phases start at model time zero, without a mission epoch. Both trajectories use LU = 384400 km and TU = 375190.25852 s. The source Gateway project used TU = 375695 s.

## Uncontrolled observer departure from a periodic reference

The physical observer follows an uncontrolled NRHO trajectory initialized from the SNRHL2 catalog state, without burns or resets. Only the nominal reference is differentially corrected to a periodic CR3BP orbit. Its period is $T$.

Departure is the observer's difference from that reference at matching phase, not its range to Gateway. The initial offset makes first-period RMS departure nonzero. The seeded ensemble adds 1-km position and 0.01-m/s velocity perturbations per axis to show sensitivity to initial conditions. Large departure motivates a guidance study but does not establish a safe maneuver deadline.

**Why use an uncontrolled trajectory?** Station-keeping is deliberately omitted to expose departure from the nominal halo and the associated changes in observation geometry and tracking performance. The growing position/velocity deviations also provide a range of cases for studying corrective ΔV requirements and maneuver feasibility under the assumed burn cap. This is a stress-test baseline. The candidate burns are not applied, so these results do not yet demonstrate closed-loop recovery.

**How to assess stability:** evaluate the corrected periodic nominal orbit, not the nonclosing catalog trajectory. Integrate the variational equations over one corrected period to obtain the monodromy matrix, then inspect its eigenvalues (Floquet multipliers). A multiplier with magnitude above one indicates a linearly unstable mode. Values on the unit circle require checks for degeneracy and nonlinear behavior. Confirm practical boundedness using small-perturbation ensembles over a stated horizon and position/velocity corridor. The uncontrolled departure plots alone do not establish a quasi-stability classification. See [NASA’s NRHO stability and orbit-maintenance analysis](https://ntrs.nasa.gov/citations/20170001347).

## Stability of the corrected nominal NRHO

This diagnostic evaluates the **corrected periodic nominal orbit**. It is separate from the uncontrolled catalog-seed departure. Integrating the variational equations

$$
\dot{\Phi}(t)=A(t)\Phi(t), \qquad \Phi(0)=I
$$

over one corrected period $T$ gives the **monodromy matrix** $M=\Phi(T)$. Its eigenvalues $\lambda_i$ are **Floquet multipliers**. Their largest magnitude is the **spectral radius**

$$
\rho(M)=\max_i |\lambda_i|.
$$

A multiplier with magnitude greater than one indicates a linearly unstable mode. In that mode, the perturbation amplitude is multiplied by $|\lambda_i|$ after each period at matching orbital phase. A negative real multiplier also reverses the mode's sign each period. This does not mean every initial perturbation or physical error grows by the same factor at every epoch.

For $\rho(M)>1$, the dominant local modal **e-folding time** is

$$
\tau=\frac{T}{\ln\rho(M)}.
$$

It is the timescale for the dominant mode's exponential envelope to grow by a factor of $e$. It is not a maneuver deadline or a prediction of total navigation error. A small **closure error** means the reference returns close to its starting state after one period. It does not establish stability of nearby trajectories.

A second integration checks sensitivity to solver tolerances. The code requires adequate reference closure and a spectral-radius margin before reporting instability. Near-unit results remain unresolved. Unit-circle multipliers alone do not prove nonlinear stability, and repeated unit multipliers require additional care.

The ensemble below starts around the **corrected nominal state**, with seeded 1-km / 0.01-m/s per-axis perturbations and no burns. Its 100-km / 1-m/s corridor is an exploratory requirement independent of the Floquet classification. Non-exits are censored at the study end. Finite-horizon boundedness does not establish nonlinear stability or a quasi-stability classification.

Reference: [NASA's geometrical analysis of station-keeping using Floquet modes](https://ntrs.nasa.gov/citations/20220005226).

In [ ]:
import numpy as np
from halo_tracking.stability import orbit_stability, save_stability
stability = orbit_stability(cfg, periods=5, ensemble_count=8)
save_stability(stability, ROOT / 'results/stability')
stab = stability['summary']
multiplier_rows = ''.join(
    f"<tr><td>{i+1}</td><td>{v['real']:+.6f}</td>"
    f"<td>{v['imag']:+.6f}</td><td>{v['magnitude']:.6f}</td></tr>"
    for i, v in enumerate(stab['floquet_multipliers']))
display(HTML('<table><tr><th>Multiplier</th><th>Real</th>'
             '<th>Imaginary</th><th>Magnitude</th></tr>' + multiplier_rows + '</table>'))
tau = stab['efolding_time_days']
timescale = f'{tau:.2f} days' if tau is not None else 'not established'
display(Markdown(
    f"**Nominal-orbit result: {stab['classification']}.** "
    f"Largest multiplier magnitude: **{stab['spectral_radius']:.6f}**. "
    f"local e-folding time: **{timescale}**. "
    f"The spectral radius changes by {stab['spectral_radius_tolerance_change']:.2e} "
    f"between solver tolerances. One-period position closure is "
    f"{stab['position_closure_km']:.2e} km. "
    "Near-unit neutral multipliers can split numerically. The classification "
    "uses a margin and a closure check, not exact equality to one."
))

fig, axes = plt.subplots(2, 1, figsize=(9.4, 6.2), sharex=True, layout='constrained')
for ax, key, limit, label in zip(
        axes, ['position_departure_km', 'velocity_departure_m_s'],
        [stab['position_limit_km'], stab['velocity_limit_m_s']],
        ['Position departure (km)', 'Velocity departure (m/s)']):
    for i, history in enumerate(stability[key]):
        ax.semilogy(stability['times_tu']/stab['period_tu'],
                    np.maximum(history, 1e-12), color='#245b80', alpha=.45,
                    label='Perturbed uncontrolled arcs' if i == 0 else None)
    ax.axhline(limit, color='#ca7139', ls='--', label='Study corridor limit')
    ax.set_ylabel(label)
    ax.grid(alpha=.17)
axes[-1].set_xlabel('Elapsed corrected nominal periods')
fig.suptitle('Sensitivity to perturbations about the periodic nominal NRHO')
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center',
           ncol=2, frameon=False)
export(fig, ROOT / 'results/stability/nominal_perturbation_departure.png')
display(fig)
plt.close(fig)
display(Markdown(
    f"**Finite-horizon result:** {stab['arcs_with_sampled_corridor_exit']} of "
    f"{stab['ensemble_count']} perturbed arcs leave the study corridor within "
    f"{stab['periods']} nominal periods. Crossings are detected at sampled epochs. "
    "These deviations help expose tracking sensitivity and supply corrective ΔV "
    "cases. They do not establish successful maneuver execution or recovery. "
    "Diagnostics and histories are saved under `results/stability/`."
))

In [ ]:
from halo_tracking.departure import reference_departure, save_departure
from halo_tracking.plotting import departure_figures
departure = reference_departure(cfg, periods=20, ensemble_count=8)
save_departure(departure, cfg, ROOT / 'results/departure')
departure_plots = departure_figures(departure, cfg)
export(departure_plots['departure'], ROOT / 'results/departure/departure.png')
export(departure_plots['period_growth'], ROOT / 'results/departure/period_growth.png')

In [ ]:
closure = departure['diagnostics']
periods = [1, 2, 4, 6, 8, 10, 20]
rows = ''.join(f"<tr><td>{k-1}T → {k}T</td><td>{departure['period_position_rms_km'][k-1]:,.1f}</td>"
               f"<td>{departure['period_velocity_rms_m_s'][k-1]:,.3f}</td></tr>" for k in periods)
display(HTML('<table><tr><th>Period interval</th><th>RMS position departure (km)</th>'
             '<th>RMS velocity departure (m/s)</th></tr>' + rows + '</table>'))
display(Markdown(f"""
The corrected **nominal** one-period closure is **{closure['position_closure_km']:.6f} km** and **{closure['velocity_closure_m_s']:.6f} m/s**. The original rounded **catalog** state's one-period closure error is **{closure['catalog_closure_km']:.1f} km**. The initial rounded truth-to-nominal position offset is **{closure['initial_nominal_offset_km']:.1f} km**.

Each RMS value covers that period alone. Departure can decrease temporarily as the geometry changes. This does not establish stabilization. The guidance section below uses exploratory trigger and corridor values, not operational limits.
"""))

display(Markdown(
    f"**Departure trend:** period RMS grows from "
    f"{departure['period_position_rms_km'][0]:,.1f} km in period 1 to "
    f"{departure['period_position_rms_km'][-1]:,.1f} km in period 20. "
    "The long-horizon motion has left the nominal halo neighborhood. "
    "it illustrates uncontrolled CR3BP divergence, not a maintained orbit."
))


## Cumulative uncontrolled observer trajectories

Each panel extends the same continuous truth propagation: $0\to T$, $0\to2T$, …, $0\to20T$. The dashed curve repeats the corrected nominal orbit at matching phase. Panel annotations give the maximum departure reached up to that horizon.

The displayed grid auto-scales to include the growing trajectory. A second exported grid uses fixed nominal-reference limits to resolve early departure. Later excursions can leave those limits.

In [ ]:
from halo_tracking.plotting import period_trajectory_grid
fig_period_grid = period_trajectory_grid(departure, cfg, scale_mode='local')
export(fig_period_grid, ROOT / 'results/departure/observer_period_grid.png')
display(fig_period_grid)
plt.close(fig_period_grid)

fig_reference_grid = period_trajectory_grid(departure, cfg, scale_mode='reference')
export(fig_reference_grid, ROOT / 'results/departure/observer_period_grid_fixed_scale.png')
plt.close(fig_reference_grid)

## Gateway prediction without new observations

Gateway truth receives a fixed-direction inertial acceleration of **2 × 10⁻⁹ m/s²**, while both EKFs use unforced CR3BP. Gateway measurements are deliberately disabled at **8T** by default. Before then, all visibility masks apply. LSO observations retain their geometric constraints. Truth and filters propagate continuously.

Compare instantaneous error, rolling 24-hour RMSE, filter-reported uncertainty, and time since the last accepted Gateway observation. The last observation can precede the imposed blackout. This is one noise realization with an illustrative disturbance, so neither case is guaranteed to have lower actual error at every epoch. Covariance excludes the unmodeled acceleration and is not a calibrated error bound.

The tables report endpoint errors and the first sampled threshold crossing at or after blackout. A threshold already exceeded at blackout is reported there. A non-crossing is right-censored at study end. These are sensitivity results, not mission prediction limits.

In [ ]:
import numpy as np
import os
observer_period_count = int(os.environ.get('HALO_GATEWAY_PERIODS', '20'))

def render_metrics_table(title, headers, records):
    """Compact, horizontally scrollable table for blind prediction results."""
    headings = ''.join(
        '<th style="padding:8px 11px;text-align:right;'
        'background:#eef3f8;border-bottom:1px solid #cbd5e1;">'
        + str(label) + '</th>' for label in headers)
    rows = ''.join(
        '<tr style="background:' + ('#f6f8fc' if i%2 else 'white') + '">'
        + ''.join('<td style="text-align:right;padding:7px 11px;'
                  'border-bottom:1px solid #e5e9f0;white-space:nowrap;">'
                  + str(value) + '</td>' for value in record)
        + '</tr>' for i, record in enumerate(records))
    return (
        '<div style="margin:16px 0 6px;font-weight:600;">' + title +
        '</div><div style="overflow-x:auto;max-height:590px;'
        'border:1px solid #d5dde7;border-radius:8px;">'
        '<table style="border-collapse:collapse;min-width:800px;'
        'font:12px system-ui,sans-serif;font-variant-numeric:tabular-nums;">'
        '<thead><tr>' + headings + '</tr></thead><tbody>' + rows +
        '</tbody></table></div>')

from halo_tracking.gateway_prediction import (
    run_gateway_prediction_study, save_gateway_prediction_study,
    load_gateway_prediction_study
)
from halo_tracking.plotting import gateway_prediction_figure

prediction_periods = observer_period_count
blackout_period = min(int(os.environ.get('HALO_GATEWAY_BLACKOUT_PERIODS', '8')),
                      prediction_periods - 1)
truth_acceleration = float(os.environ.get(
    'HALO_GATEWAY_TRUTH_ACCELERATION_M_S2', '2e-9'))
prediction_output = ROOT / 'results/gateway_prediction'
prediction_kwargs = dict(
    periods=prediction_periods, blackout_period=blackout_period,
    disturbance_m_s2=truth_acceleration, rolling_window_days=1.,
    thresholds_km=(1., 10., 100.))
prediction = load_gateway_prediction_study(
    cfg, prediction_output, **prediction_kwargs)
if prediction is None:
    prediction = run_gateway_prediction_study(cfg, **prediction_kwargs)
    save_gateway_prediction_study(prediction, prediction_output)
    print('Computed new truth-mismatch / forced-blackout prediction scenario.')
else:
    print('Loaded matching saved prediction scenario and time histories.')

fig = gateway_prediction_figure(prediction)
export(fig, prediction_output / 'blind_prediction_endurance.png')
display(fig)
plt.close(fig)

summary = prediction['summary']
display(Markdown(
    f"**Model mismatch:** {summary['disturbance_m_s2']:.2e} m/s² fixed "
    f"inertial direction. **forced blackout starts:** "
    f"{summary['blackout_elapsed_days']:.2f} days. "
    f"**last accepted Gateway observation:** "
    f"{summary['last_accepted_gateway_observation_days']} days. "
    "The estimator uses unforced CR3BP throughout."
))

cols = ['Horizon', 'New Gateway obs.', 'Days since last obs.',
        'Exact obs. error (km)', 'LSO obs. error (km)',
        'Exact obs. 24h RMSE (km)', 'LSO obs. 24h RMSE (km)']
hrows = []
for k in range(1, prediction_periods + 1):
    pair = {r['case']: r for r in prediction['rows']
            if r['horizon_periods'] == k}
    a, b = pair['perfect'], pair['lso_estimated']
    age = a['days_since_gateway_observation']
    hrows.append([f'0 → {k}T', a['gateway_observations_latest_period'],
                  f'{age:.2f}' if np.isfinite(age) else 'No observation yet',
                  f"{a['instantaneous_position_error_km']:.3f}",
                  f"{b['instantaneous_position_error_km']:.3f}",
                  f"{a['rolling_position_rmse_km']:.3f}",
                  f"{b['rolling_position_rmse_km']:.3f}"])
display(HTML(render_metrics_table(
    'Gateway error at each cumulative-horizon endpoint', cols, hrows)))

limits_rows = []
for r in prediction['threshold_rows']:
    value = r['crossing_days_since_last_observation']
    when = (f'{value:.2f}' if value is not None
            else 'Not reached by study end')
    limits_rows.append([r['case'],
                        f"{r['position_threshold_km']:g}", when])
display(HTML(render_metrics_table(
    'First time position-error tolerance is exceeded after blackout',
    ['Observer knowledge', 'Error tolerance (km)',
     'Days after last accepted Gateway observation'], limits_rows)))

final = {r['case']: r for r in prediction['rows']
         if r['horizon_periods'] == prediction_periods}
display(Markdown(
    f"**End-of-study errors:** exact-observer "
    f"{final['perfect']['instantaneous_position_error_km']:,.2f} km. "
    f"LSO-estimated observer "
    f"{final['lso_estimated']['instantaneous_position_error_km']:,.2f} km. "
    "Use the histories and threshold table to judge degradation. "
    "a temporary error decrease does not restart the prediction clock."
))


## Station-keeping system identification: learn the corrective ΔV

**Goal:** use the generated navigation dataset to determine the observer's **three-component corrective impulse**, $\Delta\mathbf{v}_k$, relative to the corrected periodic halo reference. The target is a burn in m/s, not the state velocity error or the next unforced state.

The exploratory teacher requests a burn when estimated position departure reaches **50 km** or velocity departure reaches **0.5 m/s**. It computes a regularized STM correction toward the nominal state **0.25T** later, weighting position and velocity by a **100-km / 1-m/s** study corridor. A candidate is valid only if its norm is at most **10 m/s** and nonlinear propagation reduces weighted terminal error. This check does not guarantee corridor recovery. No-trigger labels are zero. Rejected labels are NaN and must be masked.

Each independent arc spans **2T**, with four decision intervals per period (**nine epochs including both endpoints**). Optical sampling uses a 6-hour base grid plus decision epochs. Inputs include estimated and nominal states, reference-relative deviation, phase, EKF covariance, visibility, and observation age. Truth is reserved for evaluation. Candidate burns are **not applied**: these are local guidance examples along uncontrolled arcs.

### Connection to nonlinear ARX (NARX)

A proposed inverse-guidance NARX model predicts the burn from its past outputs and current/past navigation features:

$$
\widehat{\Delta\mathbf{v}}_k = f_\theta\!\left(
\widehat{\Delta\mathbf{v}}_{k-1},\ldots,\widehat{\Delta\mathbf{v}}_{k-n_a},
\mathbf{z}_k,\ldots,\mathbf{z}_{k-n_b}\right),
$$

where $\mathbf{z}_k$ contains the deployment-available features above, with phase represented by sine/cosine. This identifies a **guidance mapping**. A forward dynamics NARX would instead take applied burns as inputs and predict state evolution.

Build lag windows within each arc on the uniform decision grid. Fit preprocessing on training arcs only. Choose lag orders using validation arcs and report performance on held-out test arcs. Mask invalid targets and any required invalid lag labels. Handle missing observations and initial observation age explicitly. Compare against a current-feature regression baseline and report burn-vector error separately for triggered and no-burn cases.

During teacher-forced training, lagged labels can supply past outputs. Recursive evaluation must use the model's own past predictions. Here those outputs remain **candidate burns**, not an applied control history. Closed-loop station-keeping requires new rollouts that apply each predicted burn, update navigation, and measure subsequent departure and total ΔV. No NARX model or closed-loop controller is trained in this step.

In [ ]:
from halo_tracking.identification import generate_dataset
identification_summary = generate_dataset(
    cfg, ROOT / 'results/system_identification')
headers = ['Split', 'Arcs', 'Decisions', 'Triggered',
           'Feasible burns', 'Rejected burns', 'No-burn samples', 'LSO visibility']
rows = ''.join(
    '<tr>' + ''.join(f'<td>{value}</td>' for value in
    [name, item['arcs'], item['decisions'], item['triggered'],
     item['valid_triggered'], item['rejected'], item['no_burn'],
     f"{100*item['visible_fraction']:.1f}%"]) + '</tr>'
    for name, item in identification_summary.items())
display(HTML(
    '<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) +
    '</tr>' + rows + '</table>'))
display(Markdown(
    'Saved **training.npz**, **validation.npz**, **test.npz** and '
    '**metadata.json** to `results/system_identification/`. '
    'Training targets: **target_delta_v_rotating_m_s**. '
    'Always mask examples with **maneuver_label_valid=False**.'
))

triggered = sum(item['triggered'] for item in identification_summary.values())
rejected = sum(item['rejected'] for item in identification_summary.values())
display(Markdown(
    f"**Label coverage:** {triggered - rejected} valid burns and {rejected} rejections "
    "among triggered decisions. Valid labels demonstrate local predicted error reduction, "
    "not closed-loop station-keeping success. Each arc has only nine decisions, "
    "so long NARX lag windows leave few training examples."
))
